# Stage 1 — Data Foundation, Leakage Audit & AML Baseline Bridge
**Deep Learning Capstone — Phase 1 (Member 1: Data & Baseline Architecture)**  
**Target:** Continuous Canadian Council of Ministers of the Environment Water Quality Index (`CCME_Values` $\in [31.3, 100.0]$)  
**Objective:** Establish a zero-leakage, leak-safe data pipeline, audit formula leakage risk, bridge frozen AML benchmarks, and provide PyTorch DataLoaders for Member 2 (MLP) and Member 3 (Evaluation).

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

# Robust sys.path configuration
dl_dir = None
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / "dl" / "src" / "data.py").exists():
        dl_dir = base / "dl"
        break
    if (base / "AML_Capstone" / "dl" / "src" / "data.py").exists():
        dl_dir = base / "AML_Capstone" / "dl"
        break
    if (base / "src" / "data.py").exists() and base.name == "dl":
        dl_dir = base
        break

if dl_dir and str(dl_dir) not in sys.path:
    sys.path.insert(0, str(dl_dir))

DL_ROOT = dl_dir
from src.utils import load_config, seed_everything, resolve_path
from src.data import load_dataset, load_split_indices, verify_zero_site_leakage, load_splits, get_dataloaders, run_leakage_audit, WaterQualityPreprocessor

seed_everything(42)
cfg = load_config()
print("Config loaded successfully. Target:", cfg["data"]["target_name"])


## 1. Target Definition & Ceiling Censoring Analysis
The target is continuous `CCME_Values` bounded in $[31.3041, 100.0]$. Over 41% of ambient monitoring records reach the legal pristine upper ceiling of $100.0$.

In [ ]:
df = load_dataset(cfg)
target = df[cfg["data"]["target_name"]]

pct_100 = (target == 100.0).mean() * 100.0
print(f"Total Observations: {len(target):,}")
print(f"Target Min: {target.min():.4f}, Max: {target.max():.4f}, Median: {target.median():.4f}")
print(f"Target Observations at 100.0 (Ceiling Censored): {pct_100:.2f}%")

if "CCME_WQI" in df.columns:
    print("\nDiscrete Quality Tier Distribution:")
    print(df["CCME_WQI"].value_counts(normalize=True).mul(100).round(2).to_string())

## 2. Zero-Leakage Spatial Split Fidelity
Standard random splitting leaks repeated monitoring stations (`Area`), creating artificial memorization. We enforce strict **Group-Stratified Site-Held-Out** splitting.

In [ ]:
split_info = load_split_indices(cfg)
train_idx = np.array(split_info["train_sample_idx"])
val_idx = np.array(split_info["val_sample_idx"])
test_idx = np.array(split_info["test_sample_idx"])

leakage_stats = verify_zero_site_leakage(df, train_idx, val_idx, test_idx, group_col="Area")
split_table = [
    ["Train (Subsampled)", len(train_idx), leakage_stats["train_sites"], "0% (Disjoint)"],
    ["Val (Subsampled)", len(val_idx), leakage_stats["val_sites"], str(leakage_stats["overlap_train_val"]) + " sites"],
    ["Test (Subsampled)", len(test_idx), leakage_stats["test_sites"], str(leakage_stats["overlap_train_test"]) + " sites"]
]
print(tabulate(split_table, headers=["Split Fold", "Samples", "Unique Stations (Area)", "Overlap with Train"], tablefmt="github"))

## 3. Exploratory Data Analysis & Skewness Profiling
Environmental contaminant concentrations follow log-normal distributions with heavy right skew. `log1p` transformation stabilizes variance for neural network training.

In [ ]:
raw_features = cfg["data"]["raw_features"]
train_df = df.iloc[train_idx]

skew_data = []
for col in raw_features:
    s_raw = pd.to_numeric(train_df[col], errors="coerce").dropna()
    raw_skew = s_raw.skew()
    log_skew = np.log1p(np.maximum(0.0, s_raw)).skew()
    skew_data.append([col, round(raw_skew, 2), round(log_skew, 2), "log1p applied" if abs(raw_skew) > 1.5 else "kept linear"])

print(tabulate(skew_data, headers=["Analyte", "Raw Skewness", "log1p Skewness", "Decision"], tablefmt="github"))

## 4. Limnological Feature Engineering
We synthesize 21 domain features capturing dissolved oxygen saturation deficits ($DO_{sat} = 468 / (31.6 + T)$), stoichiometric $N:P$ nutrient ratios, and non-linear pH buffering capacity.

In [ ]:
splits_data = load_splits(cfg, scaled=False, mock=False, subsample=True)
feat_names = splits_data["feature_names"]
print(f"Total Engineered Features: {len(feat_names)}")
for i, f in enumerate(feat_names, 1):
    print(f"  {i:2d}. {f}")

## 5. Formula Leakage Audit (Task C)
Does the model simply memorize the arithmetic Canadian WQI formula ($WQI = 100 - \frac{\sqrt{F_1^2 + F_2^2 + F_3^2}}{1.732}$)?  
We benchmark linear regression on raw chemistry against all engineered features and ablated representations.

In [ ]:
audit = run_leakage_audit(cfg)
audit_table = [
    ["Linear Ridge (8 Raw Features)", audit["raw_8_chemistry"]["R2"], audit["raw_8_chemistry"]["RMSE"], audit["raw_8_chemistry"]["MAE"]],
    ["Linear Ridge (21 Engineered Features)", audit["all_21_engineered"]["R2"], audit["all_21_engineered"]["RMSE"], audit["all_21_engineered"]["MAE"]]
]
print(tabulate(audit_table, headers=["Architecture", "Test R²", "Test RMSE", "Test MAE"], tablefmt="github"))
print("\nAudit Finding:", audit["finding"])

## 6. AML Baseline Bridge & Predictions Export (Task E)
Validation of exported prediction CSVs (`results/predictions/`) and baseline metrics (`results/aml_baseline_metrics.json`) against reported AML README benchmarks.

In [ ]:
metrics_json_path = resolve_path("results/aml_baseline_metrics.json")
with open(metrics_json_path) as f:
    baseline_metrics = json.load(f)

bench_table = [
    ["XGBoost (Boundary-Clipped)", baseline_metrics["xgboost"]["val"]["RMSE"], baseline_metrics["xgboost"]["val"]["R2"], baseline_metrics["xgboost"]["test"]["RMSE"], baseline_metrics["xgboost"]["test"]["R2"]],
    ["Random Forest (100 Trees)", baseline_metrics["random_forest"]["val"]["RMSE"], baseline_metrics["random_forest"]["val"]["R2"], baseline_metrics["random_forest"]["test"]["RMSE"], baseline_metrics["random_forest"]["test"]["R2"]],
    ["XGBoost (Leakage-Aware Ablated)", baseline_metrics["xgboost_leakage_aware"]["val"]["RMSE"], baseline_metrics["xgboost_leakage_aware"]["val"]["R2"], baseline_metrics["xgboost_leakage_aware"]["test"]["RMSE"], baseline_metrics["xgboost_leakage_aware"]["test"]["R2"]]
]
print(tabulate(bench_table, headers=["Model Family", "Val RMSE", "Val R²", "Test RMSE", "Test R²"], tablefmt="github"))

## 7. Data Contract Verification for Member 2 (MLP) & Member 3 (Evaluation)
Teammates can immediately instantiate PyTorch DataLoaders with zero code modifications.

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(cfg, batch_size=256, scaled=True, mock=False)
batch_x, batch_y = next(iter(train_loader))
print(f"TrainLoader Batch X Shape: {batch_x.shape}, Dtype: {batch_x.dtype}")
print(f"TrainLoader Batch y Shape: {batch_y.shape}, Dtype: {batch_y.dtype}")
print(f"X Mean across batch: {batch_x.mean().item():.4f}, Std: {batch_x.std().item():.4f}")
print("Handoff interface to Member 2 (MLP) and Member 3 (Evaluation): VERIFIED & READY.")